<a href="https://colab.research.google.com/github/cadairhinojosa/Seeing-Through-Words---AI-Image-Captioning/blob/main/SeeingThroughWords.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [11]:
#Installed the necessary transformers and packages
!pip install transformers torch torchvision --quiet
!pip install ultralytics --quiet
!pip install gradio --quiet
!pip install gtts --quiet

In [12]:
#import transformers and packages and gTTs (google Text-Speech )
from transformers import BlipProcessor, BlipForConditionalGeneration
from PIL import Image
from ultralytics import YOLO
import gradio as gr
import torch
from gtts import gTTS
import tempfile


In [13]:
#Setting up the BlipProcessor for image captioning
processor = BlipProcessor.from_pretrained("Salesforce/blip-image-captioning-base")
model_caption = BlipForConditionalGeneration.from_pretrained("Salesforce/blip-image-captioning-base")


In [14]:
#setting up the Yolo Model for object data detection
model_yolo = YOLO("yolov8n.pt")  # Use the lightweight version


In [15]:
#creating danger keywords
danger_keywords = [
    'police','fire', 'gun', 'knife held aggressively', 'weapon', 'explosion', 'smoke',
    'car', 'truck', 'bus', 'train', 'Aggressive crowd', 'bear', 'dog being aggressive',
    'cliff', 'motorcycle', 'police car', 'rain', 'drugs','police light', 'danger sign',
    'blood', 'crime scene', 'handcuffs', 'broken', 'glass', 'ambulance', 'crime', 'fight',
]


In [16]:
#Take an image and generate a text description
def generate_caption(image):
    inputs = processor(images=image, return_tensors="pt")
    out = model_caption.generate(**inputs, no_repeat_ngram_size=2)
    return processor.decode(out[0], skip_special_tokens=True)


In [17]:
def text_to_audio(text_input):
    tts = gTTS(text=text_input, lang='en')
    temp_audio_file = tempfile.NamedTemporaryFile(delete=False, suffix='.mp3')
    temp_audio_file_path = temp_audio_file.name
    temp_audio_file.close()

    tts.save(temp_audio_file_path)
    return temp_audio_file_path

print("text_to_audio function defined successfully.")

text_to_audio function defined successfully.


In [18]:
# detect the danger key words and check if they match with crime scene words
#If else to assist with policy danger key word matching with crime scence to print proper output

def detect_danger(image):
    results = model_yolo(image)
    labels = []
    for r in results:
        labels += [model_yolo.names[int(c)] for c in r.boxes.cls]

    matched = [item for item in labels if item.lower() in danger_keywords]

    # Check for specific crime scene keywords first, excluding 'police'
    crime_scene_keywords = ['knife', 'blood', 'fight', 'handcuffs', 'gun','weapon','police']
    if any(word in matched for word in crime_scene_keywords):
        return f"🔴 Possible Crime Scene or danger: {', '.join(set(matched))}"
    # Check if 'police' is present without other crime scene keywords
    elif 'police' in matched and not any(word in matched for word in crime_scene_keywords):
        return f"🟡 Caution: {', '.join(set(matched))}"
    # Check for other danger keywords
    elif any(word in matched for word in danger_keywords if word not in crime_scene_keywords and word != 'police'):
         return f"🟡 Caution: {', '.join(set(matched))}"
    else:
        return "🟢 Safe"

In [19]:
def analyze_image(img):
    caption = generate_caption(img)
    danger = detect_danger(img)

    # Optional: flag certain keywords from the caption
    crime_words = ['arrest', 'weapon', 'blood', 'shooting', 'gun','knife']
    if any(word in caption.lower() for word in crime_words):
        danger = "🔴 Possible Crime Scene or danger (based on caption)"


    spoken_danger_status = "Unknown Safety Status"
    if danger.startswith("🔴 Possible Crime Scene"):
        spoken_danger_status = "Possible Crime Scene"
    elif danger.startswith("🟡 Caution"):
        spoken_danger_status = "Caution"
    elif danger == "🟢 Safe":
        spoken_danger_status = "Safe"

    # Combine text for narration
    combined_text = f"The image shows: {caption}. Safety status: {spoken_danger_status}"
    audio_path = text_to_audio(combined_text)

    return f"Caption: {caption}\n\nSafety Status: {danger}", audio_path

In [20]:
#Utlizing Gradio library to create a user friendly web interface
import gradio as gr
gr.Interface(
    fn=analyze_image,
    inputs=gr.Image(type="pil"),
    outputs=[gr.Textbox(label="Caption Results: ", lines=10),gr.Audio(type="filepath", label="Audio Narration")],
    title=" Seeing Through Words App",
    description="Upload an image to receive a text/audio description and a danger warning"
).launch()


It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://447fbbb4695fd492ac.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
